# Combining and Normalising WARN and fyi_layoffs
This notebook walks throgh every transformation step, so you can easily verify the output
The three sources we are merging into a single csv are:

| # | File | Description |
|---|------|-------------|
| 1 | `fyi_layoffs.csv` | Crowd-sourced tracker (global, many industries) |
| 2 | `WARNDatabase2026.csv` | US WARN-act filings for 2026 |
| 3 | `WARNDatabaseMasterExcluding2026_.csv` | US WARN-act filings up to end of 2025 |



## Step 1: Imports

In [1]:
import pandas as pd
from dateutil import parser as date_parser

# Show all columns when previewing a DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## Step 2: Load the csv files
We load the CSV with `encode_errors='replace'` to handle any non UTF-8 Chars

In [2]:
df1 = pd.read_csv('src/fyi_layoffs.csv', encoding_errors='replace')
df2 = pd.read_csv('src/WARNDatabase2026.csv', encoding_errors='replace')
df3 = pd.read_csv('src\WARNDatabaseMasterExcluding2026 .csv', encoding_errors='replace')

print("=== fyi_layoffs.csv ===")
print(f"Shape: {df1.shape} (rows x columns)" )
print(df1.columns.tolist())
display(df1.head(3))

print("\n=== WARNDatabase2026.csv ===")
print(f"Shape: {df2.shape} (rows x columns)" )
print(df2.columns.tolist())
display(df2.head(3))

print("\n=== WARNDatabaseMasterExcluding2026_.csv ===")
print(f"Shape: {df3.shape} (rows x columns)" )
print(df3.columns.tolist())
display(df3.head(3))

<>:3: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
<>:3: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
C:\Users\tstoe\AppData\Local\Temp\ipykernel_22616\3396174582.py:3: SyntaxWarning: "\W" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\W"? A raw string is also an option.
  df3 = pd.read_csv('src\WARNDatabaseMasterExcluding2026 .csv', encoding_errors='replace')


=== fyi_layoffs.csv ===
Shape: (4387, 11) (rows x columns)
['Company', 'Location HQ', '# Laid Off', 'Date', '%', 'Industry', 'Source', 'Stage', '$ Raised (m...', 'Country', 'Date Added']


,Company,Location HQ,# Laid Off,Date,%,Industry,Source,Stage,$ Raised (m...,Country,Date Added
0,Cloudflare,SF Bay Area,1100.0,7.5.2026,20%,Security,https://blog.cloudflare....,Post-I...,$332,United States,7.5.2026
1,Bill.com,SF Bay Area,709.0,7.5.2026,30%,Finance,https://www.bill.com/bl...,Post-I...,$280,United States,7.5.2026
2,Ticketmaster,Los Angeles,350.0,7.5.2026,8%,Consumer,https://hoodline.com/2...,Acquir...,NaN,United States,9.5.2026



=== WARNDatabase2026.csv ===
Shape: (1581, 13) (rows x columns)
['State', 'Company', 'City', 'Number of Workers', 'WARN Received Date', 'Effective Date', 'Closure / Layoff', 'Temporary/Permanent', 'Union', 'Region', 'County', 'Industry', 'Notes']


,State,Company,City,Number of Workers,WARN Received Date,Effective Date,Closure / Layoff,Temporary/Permanent,Union,Region,County,Industry,Notes
0,Pennsylvania,"Bush Industries, Inc.",Erie,51,05/01/2026,04/30/2026,Closure,NaN,NaN,NaN,Erie,NaN,NaN
1,Pennsylvania,Dometic,Limerick,89,05/01/2026,09/21/2026-03/31/2027,Closure,NaN,NaN,NaN,Montgomery,NaN,NaN
2,California,PET CLUB,318 N. Sunrise Blvd. Roseville CA 95661,6,05/13/2026,07/18/2026,Closure Permanent,NaN,NaN,NaN,Placer County,44-45 Retail Trade,NaN



=== WARNDatabaseMasterExcluding2026_.csv ===
Shape: (64937, 13) (rows x columns)
['State', 'Company', 'City', 'Number of Workers', 'WARN Received Date', 'Effective Date', 'Closure/Layoff', 'Temporary/Permanent', 'Union', 'Region', 'County', 'Industry', 'Notes']


,State,Company,City,Number of Workers,WARN Received Date,Effective Date,Closure/Layoff,Temporary/Permanent,Union,Region,County,Industry,Notes
0,Oregon,Vacuum Technique LLC,Clackamas,78,12/31/2025,04/30/2026-06/30/2026,Large Layoff - 10 or more workers,NaN,NaN,NaN,NaN,NaN,9415
1,North Carolina,Kroehler Furniture Co.,Conover,208,12/30/2025,12/31/2025,Closure,Permanent,NaN,NaN,Catawba,NaN,NaN
2,California,"Mare Island Dry Dock, LLC",1180 Nimitz Avenue Vallejo CA 94592,84,12/30/2025,12/29/2025,Closure Permanent,NaN,NaN,NaN,Solano County,31-33 Manufacturing,NaN


## Step 3: Data parsing for date transformation
The three files use different date formats.

| Source | Example | Format|
|--------|---------|-------|
|fyi_layoffs|7.5.2026|DD.MM.YYYY|
| WARN 2026 | 05/01/2026| MM/DD/YYYY|
| WARN Master | 12/ 31/2025| MM/DD/YYYY|

We need to normalize the date into a format, so we can sort and compare dates correctly.
We use the comman used ISO format: YYYY-MM-DD

The parser should
1. Check for `DD.MM.YYYY` pattern
2. Falls back to `dateutil.parser` for everything else
3. Return `None` if parsing fails (e.g. missing or malformed values).


In [3]:
def parse_date(raw):
    """Convert any of the three date formats to YYYY-MM-DD. Returns None on failure."""
    try:
        format = 1 if '.' in str(raw) else 2 if '/' in str(raw) else 3
        if format == 1:
            parts = str(raw).strip().split('.')
            # D.M.YYYY pattern used in layoffs.csv  (e.g. 7.5.2026 → 2026-05-07)
            if len(parts) == 3 and len(parts[2]) == 4:
                return f"{parts[2]}-{int(parts[1]):02d}-{int(parts[0]):02d}"
            # All other formats – let dateutil figure it out
            return dateparser.parse(str(raw), dayfirst=False).strftime('%Y-%m-%d')
        elif format == 2:
            # MM/DD/YYYY pattern used in WARN 2026 (e.g. 05/01/2026 → 2026-05-01)
            parts = str(raw).strip().split('/')
            if len(parts) == 3 and len(parts[2]) == 4:
                return f"{parts[2]}-{int(parts[0]):02d}-{int(parts[1]):02d}"
            # All other formats – let dateutil hopefully figure it out
            return dateparser.parse(str(raw), dayfirst=False).strftime('%Y-%m-%d')
    except Exception:
        print(f"Warning: Failed to parse date {raw!r}")
        return None

# Quick sanity-check on each format
test_cases = [('7.5.2026', 'layoffs.csv format'), ('05/01/2026', 'WARN 2026'), ('12/31/2025', 'WARN Master')]
for raw, label in test_cases:
    print(f"  {label:25s}  {raw!r:15s} → {parse_date(raw)}")


  layoffs.csv format         '7.5.2026'      → 2026-05-07
  WARN 2026                  '05/01/2026'    → 2026-05-01
  WARN Master                '12/31/2025'    → 2025-12-31


## Step 4: Normalise each source to the target schema

We extract three columns from each file, renaming them to our target schema:

|Source column| Target column | Notes|
|-------------|---------------|------|
|Company / Company / Company | Company_name | strip lead/tail whitespace|
| # Laid Off / Number of Workers / Number of Workers | layoff_size | coerce to numeric; non-numeric → NaN |
| Date / WARN Received Date / WARN Received Date | date | parsed via parse_date() above |

No rows are dropped here. rows with missing values are kept just in case we want to inspect them later.


In [4]:
# Source 1:fyi_layoffs.csv
t1 = pd.DataFrame({
    'Company_name': df1['Company'].str.strip(),
    'layoff_size': pd.to_numeric(df1['# Laid Off'], errors='coerce'),
    'date': df1['Date'].apply(parse_date),
    'source': 'fyi_layoffs.csv' # helper column to trace source back if needed
})

t2 = pd.DataFrame({
    'Company_name': df2['Company'].str.strip(),
    'layoff_size': pd.to_numeric(df2['Number of Workers'], errors='coerce'),
    'date': df2['WARN Received Date'].apply(parse_date),
    'source': 'WARNDatabase2026.csv'
})

t3 = pd.DataFrame({
    'Company_name': df3['Company'].str.strip(),
    'layoff_size': pd.to_numeric(df3['Number of Workers'], errors='coerce'),
    'date': df3['WARN Received Date'].apply(parse_date),
    'source': 'WARNDatabaseMasterExcluding2026.csv'
})

print("Rows per src after normalization:")
print(f"  fyi_layoffs.csv: {len(t1)}")
print(f"  WARNDatabase2026.csv: {len(t2)}")
print(f"  WARNDatabaseMasterExcluding2026.csv: {len(t3)}")
print(f"Total rows across all sources: {len(t1) + len(t2) + len(t3)}")

print("\nMissing values per src after normalization:")
print(f"  fyi_layoffs.csv: {t1.isnull().sum().sum()}")
print(f"  WARNDatabase2026.csv: {t2.isnull().sum().sum()}")
print(f"  WARNDatabaseMasterExcluding2026.csv: {t3.isnull().sum().sum()}")

display(t1.head(3))
display(t2.head(3))
display(t3.head(3))

Rows per src after normalization:
  fyi_layoffs.csv: 4387
  WARNDatabase2026.csv: 1581
  WARNDatabaseMasterExcluding2026.csv: 64937
Total rows across all sources: 70905

Missing values per src after normalization:
  fyi_layoffs.csv: 1515
  WARNDatabase2026.csv: 6
  WARNDatabaseMasterExcluding2026.csv: 1086


,Company_name,layoff_size,date,source
0,Cloudflare,1100.0,2026-05-07,fyi_layoffs.csv
1,Bill.com,709.0,2026-05-07,fyi_layoffs.csv
2,Ticketmaster,350.0,2026-05-07,fyi_layoffs.csv


,Company_name,layoff_size,date,source
0,"Bush Industries, Inc.",51.0,2026-05-01,WARNDatabase2026.csv
1,Dometic,89.0,2026-05-01,WARNDatabase2026.csv
2,PET CLUB,6.0,2026-05-13,WARNDatabase2026.csv


,Company_name,layoff_size,date,source
0,Vacuum Technique LLC,78.0,2025-12-31,WARNDatabaseMasterExcluding2026.csv
1,Kroehler Furniture Co.,208.0,2025-12-30,WARNDatabaseMasterExcluding2026.csv
2,"Mare Island Dry Dock, LLC",84.0,2025-12-30,WARNDatabaseMasterExcluding2026.csv


## Step 5: Stack all three sources into a singe DataFrame

we use `pd.concat` to append the rows vertically. We reset the index so we have a index from 0 till N continously. The `source` column is kept so every row can be traced back to the origin if needed.

In [5]:
combined = pd.concat([t1, t2, t3], ignore_index=True)
print(f"Total rows after concatenation: {len(combined)}")
display(combined.sample(20,random_state=42))

Total rows after concatenation: 70905


,Company_name,layoff_size,date,source
46043,"Salon Luce, LC",14.0,2018-06-18,WARNDatabaseMasterExcluding2026.csv
17549,Cushman & Wakefield U.S. Inc.,2.0,2023-05-11,WARNDatabaseMasterExcluding2026.csv
59047,Pomona Valley Hospital- Business Office,30.0,2014-03-17,WARNDatabaseMasterExcluding2026.csv
67240,"RCN Telecom Services, LLC Presidents Plaza, Bu...",33.0,2011-05-25,WARNDatabaseMasterExcluding2026.csv
2132,Affirm,500.0,2023-02-08,fyi_layoffs.csv
69112,Navistar Inc.,63.0,2010-08-02,WARNDatabaseMasterExcluding2026.csv
26357,RTI International,48.0,2020-08-12,WARNDatabaseMasterExcluding2026.csv
39933,Future Media Group,33.0,2020-03-16,WARNDatabaseMasterExcluding2026.csv
8018,CHS Inc,25.0,2025-07-10,WARNDatabaseMasterExcluding2026.csv
44767,"Maintenance Supply Headquarters, LP",51.0,2018-11-15,WARNDatabaseMasterExcluding2026.csv


## Step 6: Uppercase company names
The same company may appear differently arrcoss all sources. Uppercasing before grouping prevents those from beeing counted as seperate companies.

We can also drop rows where `company_name` is NaN -> we need a name or the row can not be used.
If we do not have a date, we can not continue ...

In [6]:
# Drop rows with no company name
before_drop = len(combined)
combined = combined.dropna(subset=['Company_name'])
combined = combined.dropna(subset=['date'])
print(f"Dropped {before_drop - len(combined)} rows with missing company name or date")

# Upercase company names for consistent matching
combined['Company_name'] = combined['Company_name'].str.upper().str.strip()

print(f" Rows remaining: {len(combined)}")
print("\nSample of cleaned data:")
display(combined[['Company_name','source','date']].sample(8, random_state=42).reset_index(drop=True))

Dropped 1 rows with missing company name or date
 Rows remaining: 70904

Sample of cleaned data:


,Company_name,source,date
0,HONOR FINANCE,WARNDatabaseMasterExcluding2026.csv,2018-06-18
1,CUSHMAN & WAKEFIELD U.S. INC. - MARINE WAY,WARNDatabaseMasterExcluding2026.csv,2023-05-11
2,POMONA VALLEY HOSPITAL- BUSINESS OFFICE,WARNDatabaseMasterExcluding2026.csv,2014-03-17
3,"RCN TELECOM SERVICES, LLC PRESIDENTS PLAZA, BU...",WARNDatabaseMasterExcluding2026.csv,2011-05-25
4,AFFIRM,fyi_layoffs.csv,2023-02-08
5,NAVISTAR INC.,WARNDatabaseMasterExcluding2026.csv,2010-08-02
6,RTI INTERNATIONAL,WARNDatabaseMasterExcluding2026.csv,2020-08-12
7,LE DISTRICT,WARNDatabaseMasterExcluding2026.csv,2020-03-16


## Step 7: Fix overlap between warn and fyi

Before we do anything else, we want to verify the overlap between WARN and FYI. Below are companies where the same `(Company_name, date)` appears in both layoffs.csv and Warn

layoffs.csv reports the company-wide total, where WARN has per-location sub-counts.

In [7]:
key_layoffs = set(zip(combined[combined['source'] == 'fyi_layoffs.csv']['Company_name'], 
                        combined[combined['source'] == 'fyi_layoffs.csv']['date']))
key_warn    = set(zip(
    combined[combined['source'].isin(['WARNDatabase2026.csv', 'WARNDatabaseMasterExcluding2026.csv'])]['Company_name'],
    combined[combined['source'].isin(['WARNDatabase2026.csv', 'WARNDatabaseMasterExcluding2026.csv'])]['date']
))
overlaps = key_layoffs & key_warn
print(f"Exact (company, date) overlaps between layoffs.csv and WARN: {len(overlaps)}")

print("\nSample of overlaps:")
for i, (company, date) in enumerate(list(overlaps)):
    print(f"  {i+1:2d}. {company} on {date}")


# Remove overlaps from combined by removing them from layoffs.csv to avoid double-counting
combined['is_overlap'] = combined.apply(lambda row: (row['Company_name'], row['date']) in overlaps, axis=1)
display(combined[combined['is_overlap'] == True])


Exact (company, date) overlaps between layoffs.csv and WARN: 24

Sample of overlaps:
   1. MICROSOFT on 2025-07-02
   2. KATERRA on 2021-06-01
   3. CHECKR on 2024-04-09
   4. MICROSOFT on 2025-09-08
   5. TRIPADVISOR on 2020-04-28
   6. INBOUND HEALTH on 2025-12-01
   7. ASPIRATION on 2023-03-24
   8. KATERRA on 2020-04-02
   9. MICROSOFT on 2025-06-02
  10. GOOGLE on 2025-02-27
  11. MICROSOFT on 2023-01-18
  12. BLOCK on 2026-02-26
  13. MALWAREBYTES on 2022-08-17
  14. PELOTON on 2022-02-08
  15. AMAZON on 2026-01-28
  16. WIX on 2023-02-15
  17. YELP on 2020-04-09
  18. FLEXPORT on 2023-01-11
  19. SEMA4 on 2022-11-14
  20. INTUIT on 2024-07-10
  21. MICROSOFT on 2025-05-13
  22. ORACLE on 2026-03-31
  23. BIG FISH GAMES on 2020-09-01
  24. SEMA4 on 2022-08-15


,Company_name,layoff_size,date,source,is_overlap
43,ORACLE,30000.0,2026-03-31,fyi_layoffs.csv,True
73,BLOCK,4000.0,2026-02-26,fyi_layoffs.csv,True
103,AMAZON,16000.0,2026-01-28,fyi_layoffs.csv,True
148,INBOUND HEALTH,NaN,2025-12-01,fyi_layoffs.csv,True
207,MICROSOFT,42.0,2025-09-08,fyi_layoffs.csv,True
278,MICROSOFT,9000.0,2025-07-02,fyi_layoffs.csv,True
297,MICROSOFT,305.0,2025-06-02,fyi_layoffs.csv,True
313,MICROSOFT,6000.0,2025-05-13,fyi_layoffs.csv,True
379,GOOGLE,NaN,2025-02-27,fyi_layoffs.csv,True
680,INTUIT,1800.0,2024-07-10,fyi_layoffs.csv,True


In [8]:
print(combined.columns.tolist())

['Company_name', 'layoff_size', 'date', 'source', 'is_overlap']


## Step 8: Remove overlapping FYI rows

To avoid double-counting, we remove rows from the FYI source that have overlaps with WARN data. WARN data is more authoritative since companies are required by law to publish it, so we keep the WARN records and discard the FYI duplicates.

In [9]:
# Count rows before filtering
before_filter = len(combined)

# Remove rows where is_overlap=True AND source='fyi_layoffs.csv'
combined_cleaned = combined[~((combined['is_overlap'] == True) & (combined['source'] == 'fyi_layoffs.csv'))].copy()

print(f"Rows before filtering: {before_filter}")
print(f"Rows after removing overlapping FYI entries: {len(combined_cleaned)}")
print(f"Removed {before_filter - len(combined_cleaned)} duplicate FYI rows")

print("\nRemaining overlap rows (all from WARN sources):")
print(combined_cleaned[combined_cleaned['is_overlap'] == True][['Company_name', 'source', 'date', 'is_overlap']].to_string())

# remove column is_overlap since it's no longer needed
combined_cleaned = combined_cleaned.drop(columns=['is_overlap'])

Rows before filtering: 70904
Rows after removing overlapping FYI entries: 70880
Removed 24 duplicate FYI rows

Remaining overlap rows (all from WARN sources):
         Company_name                               source        date  is_overlap
4879           ORACLE                 WARNDatabase2026.csv  2026-03-31        True
5227            BLOCK                 WARNDatabase2026.csv  2026-02-26        True
5666           AMAZON                 WARNDatabase2026.csv  2026-01-28        True
5668           AMAZON                 WARNDatabase2026.csv  2026-01-28        True
6138   INBOUND HEALTH  WARNDatabaseMasterExcluding2026.csv  2025-12-01        True
7297        MICROSOFT  WARNDatabaseMasterExcluding2026.csv  2025-09-08        True
8133        MICROSOFT  WARNDatabaseMasterExcluding2026.csv  2025-07-02        True
8583        MICROSOFT  WARNDatabaseMasterExcluding2026.csv  2025-06-02        True
8829        MICROSOFT  WARNDatabaseMasterExcluding2026.csv  2025-05-13        True
9932       

## Step 9 Filter out rows that are illogical, cancelled or rescieded.
1. Find rows, with cancelled or rescieded inside the company_name and remove them
2. Check if there are rows where the layoff number is less or equal than 0

## Step 10: Normalize similar company names

Companies with slight name variations should be consolidated into a single canonical name.
This step identifies and groups companies that are likely the same entity but have different names.

Strategies:
1. **Remove location/facility codes**: `(1045) SAN DIEGO LGBT...` → `SAN DIEGO LGBT...`
2. **Remove administrative prefixes**: `*UPDATE* COMPANY` → `COMPANY`, `*AMENDED* COMPANY` → `COMPANY`
3. **Fuzzy matching**: Use string similarity to find variations like `BOEING`, `BOEING COMPANY`, `THE BOEING COMPANY`
4. **Common suffix normalization**: `INC`, `INC.`, `CORP`, `CORP.`, `LLC`, etc.



In [10]:
remove_terms = ['CANCELLED','RESCINDED']
# Remove rows where Company_name contains any of the remove_terms
pattern = '|'.join(remove_terms)
before_remove = len(combined_cleaned)
combined_cleaned = combined_cleaned[~combined_cleaned['Company_name'].str.contains(pattern, case=False, na=False)]
print(f"Rows before removing cancellations/rescissions: {before_remove}")
print(f"Rows after removing cancellations/rescissions: {len(combined_cleaned)}")
print(f"Removed {before_remove - len(combined_cleaned)} rows with cancellation/rescission")
display(combined_cleaned.sample(10, random_state=42).reset_index(drop=True))

Rows before removing cancellations/rescissions: 70880
Rows after removing cancellations/rescissions: 70870
Removed 10 rows with cancellation/rescission


,Company_name,layoff_size,date,source
0,GODADDY,35.0,2023-02-09,WARNDatabaseMasterExcluding2026.csv
1,EMPLOYMENT BACKGROUND INVESTIGATIONS,82.0,2022-09-16,WARNDatabaseMasterExcluding2026.csv
2,SEASIDE HEALTH PLAN,3.0,2020-08-21,WARNDatabaseMasterExcluding2026.csv
3,VIDEOAMP,21.0,2020-04-08,fyi_layoffs.csv
4,"CHENEY BROS., INC.",128.0,2020-05-11,WARNDatabaseMasterExcluding2026.csv
5,"XPO LOGISTICS SUPPLY CHAIN, INC",58.0,2021-03-16,WARNDatabaseMasterExcluding2026.csv
6,"*UPDATE* PITTSBURGH GLASS WORKS, LLC",2.0,2018-08-01,WARNDatabaseMasterExcluding2026.csv
7,AIRGAS,87.0,2024-08-28,WARNDatabaseMasterExcluding2026.csv
8,WELLS FARGO,17.0,2022-09-22,WARNDatabaseMasterExcluding2026.csv
9,ELITE VALLEY FOODS INC - PIZZA REV,20.0,2020-04-17,WARNDatabaseMasterExcluding2026.csv


In [ ]:
from difflib import SequenceMatcher
import re

def normalize_company_name(name):
    """Remove location codes and prefixes from company names."""
    if not isinstance(name, str):
        return name
    
    # Remove location codes like (1045) at the beginning
    name = re.sub(r'^\(\d+\)\s*', '', name)
    
    # Remove facility/location descriptors in parentheses
    # But keep abbreviations like "LLC", "(NFI) NATIONAL DISTRIBUTION", etc when they're part of the name
    # Only remove pure numeric location codes
    
    # Remove administrative prefixes
    prefixes_to_remove = ['*UPDATE*', '*AMENDED*', '*UPDATE', 'UPDATE*', '*AMENDMENT*']
    for prefix in prefixes_to_remove:
        if name.startswith(prefix):
            name = name[len(prefix):].strip()
    
    # Remove "D/B/A" (Doing Business As) and keep the original name
    # Usually format is "COMPANY D/B/A OPERATING NAME", we keep the first part
    if 'D/B/A' in name:
        name = name.split('D/B/A')[0].strip()
    
    # Remove common Inc, LLC, Corp variations and standardize
    # This helps group "HOSTESS BRANDS INC" with "HOSTESS BRANDS, INC."
    name = re.sub(r',?\s*(INC\.?|LLC\.?|CORP\.?|CORPORATION|LIMITED|LTD\.?|CO\.?|COMPANY)$', '', name, flags=re.IGNORECASE)
    
    return name.strip()

# Test the normalization function
test_names = [
    "(1045) SAN DIEGO LGBT COMMUNITY CENTER",
    "*UPDATE* ARROW INTERNATIONAL INCORPORATED/TELEFLEX",
    "HOSTESS BRANDS, INC.",
    "HOSTESS BRANDS INC",
    "HOSTESS BRANDS",
    "(MARFRED) AMCOR PACKAGING DISTRIBUTION",
    "BOEING COMPANY",
    "THE BOEING COMPANY",
]

print("Company Name Normalization Examples:")
print("-" * 80)
for name in test_names:
    normalized = normalize_company_name(name)
    print(f"  {name:50s} → {normalized}")


In [ ]:
def similarity_ratio(str1, str2):
    """Calculate similarity ratio between two strings (0.0 to 1.0)."""
    return SequenceMatcher(None, str1, str2).ratio()

def find_similar_companies(company_list, threshold=0.85):
    """
    Find groups of similar company names.
    Returns a dictionary mapping each company name to its canonical (grouped) name.
    """
    company_list = sorted(set(company_list))
    mapping = {}
    grouped = set()
    
    for i, company1 in enumerate(company_list):
        if company1 in grouped:
            continue
            
        canonical_name = company1
        similar_group = [company1]
        
        for company2 in company_list[i+1:]:
            if company2 in grouped:
                continue
            
            similarity = similarity_ratio(company1, company2)
            
            # If similar enough, add to group
            if similarity >= threshold:
                similar_group.append(company2)
                grouped.add(company2)
        
        # Map all similar companies to the first (canonical) name
        for company in similar_group:
            mapping[company] = canonical_name
    
    return mapping

# First, apply basic normalization to all company names
print("Applying basic normalization (remove prefixes, location codes, etc.)...")
combined_cleaned['Company_name_normalized'] = combined_cleaned['Company_name'].apply(normalize_company_name)

# Check how many companies were consolidated by this step
unique_before = combined_cleaned['Company_name'].nunique()
unique_after_norm = combined_cleaned['Company_name_normalized'].nunique()
print(f"\nCompanies before normalization: {unique_before}")
print(f"Companies after basic normalization: {unique_after_norm}")
print(f"Companies consolidated by basic rules: {unique_before - unique_after_norm}")

print("\nExamples of normalized names:")
sample_df = combined_cleaned[['Company_name', 'Company_name_normalized']].drop_duplicates().head(15)
display(sample_df)


In [ ]:
print("\nFinding similar company names using fuzzy matching (threshold=0.85)...")
print("This step is slow because it compares every company against every other company...")
print("(Please wait, this may take a minute or two...)\n")

unique_normalized_companies = sorted(combined_cleaned['Company_name_normalized'].unique())
print(f"Total unique normalized company names: {len(unique_normalized_companies)}")

# Find similar companies using fuzzy matching
# Using a high threshold (0.85) to minimize false positives
similarity_mapping = find_similar_companies(unique_normalized_companies, threshold=0.85)

# Count how many were grouped
groups_created = len(set(similarity_mapping.values()))
print(f"Fuzzy matching created {groups_created} canonical groups")
print(f"Reduction from {len(unique_normalized_companies)} to {groups_created} groups")

# Show examples of groups where 2+ companies were merged
print("\nExamples of grouped similar companies:")
from collections import defaultdict
reverse_mapping = defaultdict(list)
for key, val in similarity_mapping.items():
    reverse_mapping[val].append(key)

count = 0
for canonical_name, similar_names in sorted(reverse_mapping.items()):
    if len(similar_names) > 1:
        count += 1
        print(f"\n  {count}. Canonical: {canonical_name}")
        for name in similar_names:
            if name != canonical_name:
                print(f"       → {name}")
        if count >= 20:  # Show first 20 groups
            print(f"\n  ... and {len([x for x in reverse_mapping.values() if len(x) > 1]) - 20} more groups")
            break


In [ ]:
# Apply the canonical mapping to create final company names
combined_cleaned['Company_name_canonical'] = combined_cleaned['Company_name_normalized'].map(similarity_mapping)

# Use the canonical name, fallback to normalized if for some reason mapping is missing
combined_cleaned['Company_name'] = combined_cleaned['Company_name_canonical'].fillna(combined_cleaned['Company_name_normalized'])

# Drop the temporary columns
combined_cleaned = combined_cleaned.drop(columns=['Company_name_normalized', 'Company_name_canonical'])

print("\n" + "="*80)
print("STEP 10 SUMMARY - Company Name Consolidation")
print("="*80)
print(f"Unique companies after consolidation: {combined_cleaned['Company_name'].nunique()}")
print(f"Total rows: {len(combined_cleaned)}")

# Show before/after comparison
print("\nBefore/After consolidation samples:")
print("\nCompanies consolidated FROM (sample examples):")

# Show examples of big consolidations
company_counts_after = combined_cleaned['Company_name'].value_counts()
for company, count in company_counts_after.head(10).items():
    print(f"  {company:50s} ({count:3d} records)")

# Show the final result
print("\nFinal consolidated dataset:")
display(combined_cleaned[['Company_name', 'source', 'date', 'layoff_size']].sample(10, random_state=42))


In [11]:


# Sort by company name and date for easier browsing
combined_cleaned = combined_cleaned.sort_values(by=['Company_name', 'date']).reset_index(drop=True)

# save cleaned combined dataset for future use
combined_cleaned.to_csv('cleaned_layoffs.csv', index=False)